# AgentBench — LTP 환경 재현: 모델 5개 비교

**논문**: Liu et al., *AgentBench: Evaluating LLMs as Agents*, ICLR 2024 (Lateral Thinking Puzzles)

Colab에서 **런타임 유형: GPU (T4)**로 실행하세요.

| 모델 | 역할 | 평균 GP |
|---|---|---|
| llama-3.1-8b | 저비용 오픈소스 하한선 | 0.0% |
| vicuna-13b-local | 로컬 오픈소스 비교군 | 3.3% |
| claude-sonnet-5 | 타 벤더 검증 후보 | 3.8% |
| gpt-3.5-turbo | 최저 비용 기준선 | 5.8% |
| gpt-4o | 신뢰성 상한선 | 7.9% |

지표(논문 Appendix F): GP=Game Progress(메인), SGA=Single Game Accuracy. 호스트/채점 LLM은 다섯
실험 모두 gpt-3.5 고정(공정 비교). 실험은 dev(20문제) 서브셋 기준, 5개 모델 전부 완주(n=20).


## 0. 환경 설정

In [ ]:
# Colab에서 실행 시: 저장소를 클론합니다.
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_URL = "https://github.com/gksmfly/Agentbench-Mini-Eval.git"
    if not os.path.exists("Agentbench-Mini-Eval"):
        !git clone {REPO_URL}
    %cd Agentbench-Mini-Eval/project_ltp
    !pip install -q requests pandas matplotlib numpy transformers accelerate bitsandbytes
else:
    # 로컬(맥/리눅스)에서 실행 시: 이미 project_ltp/notebooks 안에 있다고 가정
    os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None

In [ ]:
import json
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

DATA_FILE = ROOT / "data" / "sample_cases.jsonl"
GOLD_FILE = ROOT / "data" / "ltp_dev_gold.jsonl"
METRICS_FILE = ROOT / "results" / "metrics.csv"

samples = [json.loads(l) for l in open(DATA_FILE)]
print(f"toy 샘플 {len(samples)}개 로드 완료")

def get_key(name: str, required: bool = True) -> str:
    """우선순위: 이미 설정된 환경변수 -> Colab Secrets(발표 전에 미리 등록) -> getpass 수동 입력"""
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    if required:
        return getpass(f"{name}: ")
    return getpass(f"{name} (없으면 그냥 Enter): ")

# 발표 전 미리 준비해두는 걸 추천:
#   Colab 왼쪽 사이드바(열쇠 아이콘) -> Secrets -> OPENAI_API_KEY / GROQ_API_KEY / ANTHROPIC_API_KEY 등록 + "노트북 액세스" 허용
os.environ["OPENAI_API_KEY"] = get_key("OPENAI_API_KEY")
groq_key = get_key("GROQ_API_KEY", required=False)
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
anthropic_key = get_key("ANTHROPIC_API_KEY", required=False)
if anthropic_key:
    os.environ["ANTHROPIC_API_KEY"] = anthropic_key

## 1. 핵심 아이디어 최소 재현 — 호스트 역할 라이브 데모 (모델 4개)

실제 dev 서브셋(20문제) 본실험은 THUDM/AgentBench 프레임워크(CLI+Docker)로 25라운드 호스트/솔버
루프를 돌렸고 (섹션 2 참고), 여기서는 그 중 **핵심 한 스텝**만 떼어냅니다: LLM에게 스토리+진실을
알려주고 "호스트" 역할을 맡긴 뒤, 솔버의 질문 하나에 "Yes"/"No"/"Irrelevant"로 답하게 합니다.

- `gpt-3.5-turbo`, `gpt-4o` → OpenAI API 호출
- `llama-3.1-8b-instant` → Groq API 호출 (본실험에 llama 계열로 실제 사용한 것과 동일 경로)
- `claude-sonnet-5` → Anthropic Messages API 호출

`vicuna-13b-local`은 API가 아니라 로컬 GPU 로드로 돌린 모델이라 이 API 라이브 데모에는 포함하지
않았습니다 — Colab GPU에 직접 로드해서 돌려보는 보너스 데모는 섹션 1.1 참고.

In [ ]:
from baseline import build_host_history, call_llm

def demo(index: int, model: str):
    entry = samples[index]
    question = entry["sample_question"]
    history = build_host_history(entry, question)
    reply = call_llm(model, history)
    print(f"--- {model} ---\n{reply}\n")

entry = samples[0]
print(f"[puzzle index] {entry['index']}")
print(f"[story] {entry['story']}\n")
print(f"[question put to the host] {entry['sample_question']}\n")

for model in ["gpt-3.5-turbo", "gpt-4o", "llama-3.1-8b-instant", "claude-sonnet-5"]:
    demo(0, model)

print(f"[gold host answer for reference, captured live by the framework] {entry.get('gold_host_answer')}")

### 1.1 보너스: 오픈소스 모델(vicuna-13b)을 API 없이 Colab GPU에 직접 로드

본실험에서 `vicuna-13b-local`은 실제로 이렇게(로컬 GPU에 직접 올려서) 실행했습니다. Colab의 무료
T4 GPU에서도 4비트 양자화로 올릴 수 있습니다 — "오픈소스 = 내 GPU만 있으면 완전 무료"라는 걸
실제로 보여주는 선택 데모입니다 (본실험 채점에는 안 씀, 섹션 2의 실제 결과와는 별개).

`lmsys/vicuna-13b-v1.5`는 게이트 모델이 아니라 별도 라이선스 동의 없이 바로 받을 수 있습니다.
번거로우면 건너뛰어도 됩니다 (섹션 1의 API 데모로 핵심은 충분히 보여줍니다).

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(없음, CPU로 진행 시 매우 느릴 수 있음)")

RUN_LOCAL_GPU_DEMO = False  # True로 바꾸면 아래 셀 실행 (다운로드 용량 큼, T4에서도 몇 분 걸릴 수 있음)

In [ ]:
if RUN_LOCAL_GPU_DEMO:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    MODEL_ID = "lmsys/vicuna-13b-v1.5"
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")

    # vicuna-13b-v1.5는 tokenizer_config.json에 chat_template이 없어서 apply_chat_template()이
    # 에러를 냅니다 -- FastChat의 공식 vicuna_v1.1 템플릿으로 프롬프트를 직접 조립합니다.
    def build_vicuna_prompt(messages):
        system = ("A chat between a curious user and an artificial intelligence assistant. "
                  "The assistant gives helpful, detailed, and polite answers to the user's questions.")
        prompt = system + " "
        for m in messages:
            role = "USER" if m["role"] == "user" else "ASSISTANT"
            prompt += f"{role}: {m['content']}" + (" " if role == "USER" else "</s>")
        return prompt + "ASSISTANT:"

    entry = samples[0]
    question = entry["sample_question"]
    chat = build_host_history(entry, question)
    prompt = build_vicuna_prompt(chat)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
else:
    print("RUN_LOCAL_GPU_DEMO = False — 건너뜀 (섹션 2에 실제 vicuna-13b-local 본실험 결과가 반영되어 있음)")


## 2. 실제 dev 서브셋(20문제) 본실험 결과 — 모델 5개 비교

각 모델을 THUDM/AgentBench 프레임워크(CLI+Docker, 25라운드 호스트/솔버 루프)로 dev 서브셋
전체에 대해 실행한 결과입니다. 아래 표는 `results/raw/LTP_runs_<model>.jsonl` 원본 로그를
직접 읽어서 만듭니다 — project_db와 동일하게, 중간 요약 파일 없이 raw 로그가 유일한 출처입니다.
`src/evaluate.py`가 GP/SGA/RE(Round Efficiency)/QR(Query Relevance)을 raw transcript에서
독립적으로 재계산해 프레임워크가 로그에 남긴 값과 대조했고, 5개 모델 전부 일치함을 확인했습니다.


In [ ]:
def load_raw_as_frame(model_id: str) -> pd.DataFrame:
    rows = []
    with open(ROOT / "results" / "raw" / f"LTP_runs_{model_id}.jsonl") as f:
        for line in f:
            d = json.loads(line)
            result = d["output"]["result"]
            rows.append({
                "index": d["index"],
                "status": d["output"]["status"],
                "GP": result["progress"],
                "SGA": result["accuracy"],
            })
    return pd.DataFrame(rows)

MODEL_IDS = ["llama-3.1-8b", "vicuna-13b-local", "claude-sonnet-5", "gpt-4o", "gpt-3.5-turbo"]
frames = {m: load_raw_as_frame(m) for m in MODEL_IDS}
order = MODEL_IDS  # already sorted by ascending GP (see summary below)

summary = []
for name in order:
    df = frames[name]
    summary.append({
        "model": name,
        "games": len(df),
        "Game Progress (%)": round(df.GP.mean()*100, 1),
        "SGA (%)": round(df.SGA.mean()*100, 1),
        "task limit reached": int((df.status == "task limit reached").sum()),
        "completed": int((df.status == "completed").sum()),
    })
summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
colors5 = ["#888780", "#C9A97E", "#7B5EA7", "#1D9E75", "#378ADD"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) GP & SGA per model, grouped
gp_sga = pd.DataFrame({
    "GP": [frames[m].GP.mean() * 100 for m in order],
    "SGA": [frames[m].SGA.mean() * 100 for m in order],
}, index=order)
gp_sga.plot.bar(ax=axes[0], color=["#1D9E75", "#7B5EA7"])
axes[0].set_title("Game Progress & SGA by model")
axes[0].set_ylabel("%")
axes[0].tick_params(axis="x", rotation=15)

# (b) completion status breakdown per model
tle = [int((frames[m].status == "task limit reached").sum()) for m in order]
comp = [int((frames[m].status == "completed").sum()) for m in order]
x = range(len(order))
axes[1].bar(x, tle, label="task limit reached (라운드 초과)", color="#D85A30")
axes[1].bar(x, comp, bottom=tle, label="completed", color="#1D9E75")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(order, rotation=15)
axes[1].set_ylabel("게임 수")
axes[1].set_title("Run status breakdown")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
# 주: LTP는 "completed"여도 GP=0일 수 있음(정답 키포인트 미달로 라운드 종료). GP가 실질 지표.


### 2.1 추가 분석: 비용·속도·라운드사용량·응답 길이

GP/SGA(정확도) 이외의 실사용 관점 지표입니다. `src/analyze.py`가 raw 로그에서 계산해서
저장해둔 `results/analysis.csv`를 읽어옵니다 — 비용은 `tiktoken`으로 토큰 수를 센 추정치,
속도는 로그 타임스탬프 차이, 라운드사용량은 게임당 실제 사용한 라운드 수(최대 25)입니다.


In [ ]:
ANALYSIS_FILE = ROOT / "results" / "analysis.csv"
analysis = pd.read_csv(ANALYSIS_FILE)

key_metrics = [
    "estimated_cost_usd_dev_subset",
    "avg_rounds_used",
    "games_per_minute",
    "avg_agent_response_chars",
]
ltp_summary = analysis[analysis["metric"].isin(key_metrics)].pivot(
    index="metric", columns="model", values="value"
).reindex(order, axis=1)
display(ltp_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ltp_summary.loc[["estimated_cost_usd_dev_subset"]].T.plot.bar(ax=axes[0], legend=False, color="darkorange")
axes[0].set_title("Estimated cost (dev subset, USD)")
axes[0].tick_params(axis="x", rotation=15)

ltp_summary.loc[["avg_rounds_used"]].T.plot.bar(ax=axes[1], legend=False)
axes[1].set_title("Average rounds used (max 25)")
axes[1].set_ylim(0, 25)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


## 3. 결론

| 모델 | GP | SGA | n | 완료 / 라운드초과 |
|---|---|---|---|---|
| gpt-3.5-turbo | 5.8% | 20.6% | 20 | 5 / 15 |
| gpt-4o | **7.9%** | **43.2%** | 20 | 1 / 19 |
| claude-sonnet-5 | 3.8% | 17.2% | 20 | **13 / 7** |
| vicuna-13b-local | 3.3% | 7.4% | 20 | 0 / 20 |
| llama-3.1-8b | 0.0% | 6.2% | 20 | 0 / 20 |

5개 모델 전부 dev 서브셋(20문제) 완주 — 표본 크기 불일치 없음.

**핵심 발견:**
- **전 모델이 GP 10% 미만** → 논문 결론("LLM은 아직 LTP를 실용적으로 못 푼다")이 5개 모델 전부에서 재현됨. 가격·벤더와 무관하게 수평추론 자체가 여전히 어려운 문제.
- **GP·SGA 둘 다 gpt-4o가 1위**(GP 7.9%, SGA 43.2%) — "질문도 잘하고 결국 진실에 가장 가깝게 다가간다."
- **claude-sonnet-5는 완료율이 가장 높음(65%, 13/20)**이지만 GP는 3.8%로 중위권 — project_db(DB 환경)에서도 claude-sonnet-5가 완료율(97%)로 압도적이었던 것과 같은 패턴. 다만 "완료"가 곧 "정답"은 아니라서(GP가 낮아도 라운드를 다 못 채우고 포기하듯 답을 냈을 수 있음) 완료율 우위가 GP 우위로 이어지지 않음.
- **오픈소스 계열(llama-3.1-8b, vicuna-13b-local)은 GP 3.3% 이하로 여전히 하위권** — 다만 vicuna-13b-local이 llama-3.1-8b보다는 나음.

**한계/주의**: dev(20문제) 서브셋 기준. claude-sonnet-5·gpt-4o는 gpt-3.5-turbo/llama-3.1-8b/vicuna-13b-local과 출시 시기가 달라(2~3년 세대 차이) "벤더/개방성 차이"와 "세대 차이"가 섞여 있음 — DB 환경에서와 동일한 한계. 채점 로직 자체는 원본 프레임워크 로그와 독립 재계산이 일치함(섹션 2 참고).
